In [40]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import torch
from torch.optim import Adam
import sympy
import os

In [56]:
## Helper Functions

# Function to convert DataFrame to array
def df_to_array(optimized_ratings):
    arr = np.concatenate([
        optimized_ratings["Offensive Rating"].to_numpy(), 
        optimized_ratings["Defensive Rating"].to_numpy()
    ])
    return arr

# Function to check if a value can be converted to a float
def is_float(value):
    try:
        float(value)
        return True
    except ValueError:
        return False

In [59]:
def get_avgRating(data):
    total_points = 0
    total_poss = 0
    for i in range(len(data)):

        try:
            # Try converting to a float
            points = float(data.iloc[i]['Tm'])
            poss = (float(data.iloc[i]['oFGA']) - float(data.iloc[i]['oORB'])
                           + float(data.iloc[i]['oTOV']) + 0.475 * float(data.iloc[i]['oFTA']))
            
            total_points += points
            total_poss += poss
        except ValueError as e:
            # Print the row that failed, and the specific error
            print(f"Skipping row {i} due to error: {e}")
            continue

    avg_OE = 100 * (total_points / total_poss)

    return avg_OE


In [62]:
## Creating Power Rankings using Adam optimizer and GPU acceleration

# Error function using PyTorch tensors
def error_function():
    team_1_indices = game_ratings[:, 0].long()
    team_2_indices = game_ratings[:, 1].long()
    valid_mask = team_2_indices != -1

    team_1_indices = team_1_indices[valid_mask]
    team_2_indices = team_2_indices[valid_mask]
    game_or = game_ratings[valid_mask, 2]
    game_dr = game_ratings[valid_mask, 3]

    predicted_ratings = (
        OR[team_1_indices] - avgRating
        + DR[team_2_indices] - avgRating
        + avgRating
    )

    total_error = torch.abs(game_or - predicted_ratings).mean()

    return total_error

# List of columns to check for float conversion
columns = ['Tm', 'Opp', 'FG', 'FGA', 'FG%', '3P', '3PA', '3P%', 'FT', 'FTA', 'FT%', 'ORB', 'TRB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'oFG', 'oFGA', 'oFG%', 'o3P', 'o3PA', 'o3P%', 'oFT', 'oFTA', 'oFT%', 'oORB', 'oTRB', 'oAST', 'oSTL', 'oBLK', 'oTOV', 'oPF']

for i in range(11):
    year = 2014 + i

    if os.path.exists(str(year) + "Ratings_Adam.csv"):
        continue

    # Reading in 2024 Game Data
    data = pd.read_csv("GameData_CSVs/" + str(year) + "GameData.csv")
    data = data[data.isna().sum(axis=1) <= 1]
    for col in columns:
        data = data[data[col].apply(is_float)]

    # Getting only team names
    team_names = sorted(list(set(data['Team'])))
    num_teams = len(team_names)

    # Creating team name to team number mapping
    team_numbers = {team: index for index, team in enumerate(team_names)}

    # Initialize ratings DataFrame with avgRating function
    avgRating = get_avgRating(data)

    optimized_ratings = pd.DataFrame({"Offensive Rating": [avgRating] * num_teams, "Defensive Rating": [avgRating] * num_teams}, index=team_names)

    # Prepare game ratings using PyTorch tensors
    game_ratings = []

    for i in tqdm(range(len(data)), desc="Calculating " + str(year) + " Game Rating", unit="game"):
        row = data.iloc[i].apply(lambda x: float(x) if isinstance(x, str) and x.replace('.', '', 1).isdigit() else x)

        if sum(isinstance(value, float) for value in row) == 0 or row.isnull().sum() > 2:
            continue

        team_1 = row["Team"]
        team_2 = row["Opponent"]
        OR1 = 100 * (row["Tm"] / (row['FGA'] - row['ORB'] + row['TOV'] + 0.475 * row['FTA']))
        OR2 = 100 * (row["Opp"] / (row['oFGA'] - row['oORB'] + row['oTOV'] + 0.475 * row['oFTA']))

        game_ratings.append([team_numbers[team_1], team_numbers.get(team_2, -1), OR1, OR2])

    # Convert game ratings list to contiguous PyTorch tensor
    game_ratings = torch.tensor(game_ratings, dtype=torch.float32, device='cuda')

    # Initialize ratings as PyTorch tensors with requires_grad=True
    OR = torch.tensor([avgRating] * num_teams, dtype=torch.float32, requires_grad=True, device='cuda')
    DR = torch.tensor([avgRating] * num_teams, dtype=torch.float32, requires_grad=True, device='cuda')

    # Create the Adam optimizer
    optimizer = Adam([OR, DR], lr=0.05)

    # Training loop
    num_epochs = 1000
    for epoch in tqdm(range(num_epochs), desc="Optimizing " + str(year) + " Ratings", unit="epoch"):
        optimizer.zero_grad()
        loss = error_function()
        loss.backward()
        optimizer.step()

    # Update optimized ratings in the DataFrame
    optimized_ratings["Offensive Rating"] = OR.detach().cpu().numpy()
    optimized_ratings["Defensive Rating"] = DR.detach().cpu().numpy()

    # Save the updated DataFrame to a CSV
    optimized_ratings.to_csv(str(year) + "Ratings_Adam.csv", index=True)

Optimizing 2024 Ratings: 100%|██████████| 1000/1000 [00:02<00:00, 495.69epoch/s]


In [64]:
## Creating Net Rankings

for i in range(11):
    year = 2014 + i

    ratings = pd.read_csv("Ratings/" + str(year) + "Ratings_Adam.csv")

    # Rename the first column to "Team"
    ratings.rename(columns={ratings.columns[0]: 'Team'}, inplace=True)

    ratings['Net Rating'] = ratings['Offensive Rating'] - ratings['Defensive Rating']

    ratings['Offensive Rank'] = ratings['Offensive Rating'].rank(ascending=False, method='min').astype(int)
    ratings['Defensive Rank'] = ratings['Defensive Rating'].rank(ascending=True, method='min').astype(int)
    ratings['Net Rank'] = ratings['Net Rating'].rank(ascending=False, method='min').astype(int)

    # off_ratings = ratings.sort_values(by='Offensive Rating', ascending=False)
    # def_ratings = ratings.sort_values(by='Defensive Rating')
    net_ratings = ratings.sort_values(by='Net Rating', ascending=False).reset_index(drop=True)

    # Reorder columns to make "Net Rank" the first column
    columns_order = ['Net Rank'] + [col for col in net_ratings.columns if col != 'Net Rank']
    net_ratings = net_ratings[columns_order]

    # off_ratings.to_csv(str(year) + "Offensive_Ratings.csv")
    # def_ratings.to_csv(str(year) + "Defensive_Ratings.csv")
    net_ratings.to_csv(str(year) + "Rankings.csv", index=False)